# Ray RLlib: Offline BC from logged CartPole trajectories

Project path: `projects/offline-marwil`

Learn from **logged experience** (no online exploration during the BC phase) — companion step 5.

1. **Record** — short PPO behavior policy → episode Parquet  
2. **Train** — Behavior Cloning from those files + online evaluation  

**Setup (once)** from the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r projects/offline-marwil/requirements.txt
```

Select that kernel. Run cells in order. Pipeline twin: `python projects/offline-marwil/run_pipeline.py`.

**BYO logs:** skip the record cell and set `DATA_DIR` in the train cell to your Parquet directory (see [README](README.md#bring-your-own-parquet-logs)).

## 1. Record behavior logs

Trains a short CartPole PPO policy, then writes evaluation episodes under `data/cartpole/`.

In [ ]:
from pathlib import Path
import sys

# Allow importing project scripts when the kernel cwd is the repo root or this folder.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "record_cartpole_logs.py").exists():
    PROJECT_DIR = Path.cwd() / "projects" / "offline-marwil"
sys.path.insert(0, str(PROJECT_DIR.resolve()))

from record_cartpole_logs import main as record_main

record_main()

## 2. Reset Ray between phases

Frees EnvRunners from the record phase so offline Ray Data is not CPU-starved.

In [ ]:
import ray

if ray.is_initialized():
    ray.shutdown()
print("Ray shut down — ready for offline training.")

## 3. Train offline (BC)

Default input is the Parquet folder written above. For bring-your-own logs, change `DATA_DIR`.

In [ ]:
import logging
import math
import os
import warnings
from pathlib import Path
from typing import Any

warnings.filterwarnings(
    "ignore",
    message=r".*RLModule\(config=\[RLModuleConfig object\]\).*",
    category=DeprecationWarning,
)

os.environ.setdefault("RAY_DATA_DISABLE_PROGRESS_BARS", "1")
logging.getLogger("ray.data").setLevel(logging.ERROR)

from ray.rllib.algorithms.bc import BCConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data").exists() and (Path.cwd() / "projects" / "offline-marwil").exists():
    PROJECT_DIR = Path.cwd() / "projects" / "offline-marwil"

# BYO: set this to your episode Parquet directory instead of the recorded default.
DATA_DIR = PROJECT_DIR / "data" / "cartpole"
TRAIN_ITERS = 20

if not DATA_DIR.exists() or not any(DATA_DIR.rglob("*")):
    raise FileNotFoundError(
        f"No offline data under {DATA_DIR}. Run the record cell first, or point DATA_DIR at your Parquet."
    )

input_uri = f"local://{DATA_DIR.resolve().as_posix()}"
print(f"[offline] training BC from {input_uri}")


def episode_return_mean(result: dict[str, Any]) -> float | None:
    env_runners = result.get("env_runners") or {}
    value = env_runners.get("episode_return_mean")
    if value is None:
        return None
    value_f = float(value)
    return None if math.isnan(value_f) else value_f


config = (
    BCConfig()
    .environment("CartPole-v1")
    .env_runners(num_env_runners=0)
    .learners(num_learners=0)
    .training(lr=1e-3, train_batch_size_per_learner=2000)
    .rl_module(
        model_config=DefaultModelConfig(
            fcnet_hiddens=[64, 64],
            fcnet_activation="tanh",
            vf_share_layers=True,
        )
    )
    .offline_data(
        input_=[input_uri],
        input_read_episodes=True,
        input_read_batch_size=256,
        map_batches_kwargs={"concurrency": 1, "num_cpus": 1},
        iter_batches_kwargs={
            "prefetch_batches": 0,
            "local_shuffle_buffer_size": None,
        },
        dataset_num_iters_per_learner=5,
    )
    .evaluation(
        evaluation_interval=1,
        evaluation_num_env_runners=0,
        evaluation_duration=8,
        evaluation_duration_unit="episodes",
        evaluation_parallel_to_training=False,
        evaluation_config=BCConfig.overrides(explore=False),
    )
    .debugging(log_level="ERROR")
)

algo = config.build_algo()
try:
    for i in range(1, TRAIN_ITERS + 1):
        result = algo.train()
        eval_block = result.get("evaluation") or {}
        eval_runners = eval_block.get("env_runners") or {}
        eval_ret = episode_return_mean({"env_runners": eval_runners})
        if eval_ret is not None:
            print(f"iter={i}  evaluate_episode_return_mean={eval_ret:.1f}")
        else:
            print(f"iter={i}  evaluate_episode_return_mean=n/a")
finally:
    algo.stop()

## What this teaches

| Idea | In this notebook |
| --- | --- |
| Offline / imitation | Train from Parquet logs, not live exploration |
| BC vs MARWIL | This smoke uses `BCConfig`; swap to `MARWILConfig` + `beta>0` for advantage weighting |
| Ops analogy | Improve a policy from historical controller logs before enabling live RL |

Platform notes: [deploy multi-worker](../../deploy/README.md) · [MLflow](../../deploy/mlflow.md) · [Project README](README.md)